# Fundamentos de Machine Learning — Evaluación Parcial 3
## Caso de Estudio: Predicción del Rendimiento de Jugadores de eSports (Regresión)

**Estudiante:** Héctor Aguila  
**Metodología:** CRISP-DM  
**Objetivo:** Desarrollar, evaluar y desplegar modelos de regresión para predecir el score de rendimiento (`performance_score`) de los jugadores de eSports.

---
# 1 y 2. Fase: Comprensión del Negocio y los Datos

### 1.1. Contexto y Objetivos del Negocio
En el entorno altamente competitivo de los eSports, el rendimiento de un jugador está determinado por múltiples factores fisiológicos y tácticos (Kills, Assists, Deaths, precisión, tiempo de reacción, fatiga, etc.). El objetivo de este proyecto es predecir de forma continua el **desempeño de rendimiento (Target: `performance_score`)** de un jugador en base a sus estadísticas en partida. Esto permite:
- Tomar decisiones tácticas de sustitución o entrenamiento en base a predicciones en tiempo real.
- Identificar qué variables (features) impactan más significativamente en el desempeño.

### 1.2. Inspección y Selección de Variables
Basándonos en la exploración y validación realizadas en la Evaluación 2:
- **Target:** `performance_score` (numérico continuo).
- **Variables Administrativas/Ruido:** Descartamos `player_id` y `record_id` porque no aportan información predictiva real.
- **Sesgo Crítico:** Descartamos `match_outcome` debido a que presentaba casi un 100% de victorias, lo que introduce un sesgo inaceptable en modelos predictivos.

In [ ]:
# 1. Importación de Librerías Core
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y Modelamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Evaluación
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Interactividad
import ipywidgets as widgets
from IPython.display import display, clear_output

import warnings
warnings.filterwarnings('ignore')

print("Librerías cargadas exitosamente.")

In [ ]:
# 2. Carga del Dataset
dataset_path = 'esports_player_performance_tournament_analytics.csv'
df = pd.read_csv(dataset_path)

print(f"Dimensiones iniciales del dataset: {df.shape[0]} registros, {df.shape[1]} columnas.")
df.head()

---
# 3. Fase: Preparación de Datos (Data Preparation)

### 3.1. Filtro Neurofisiológico de Tiempo de Reacción
Según la IAAF y estudios neurofisiológicos, el tiempo de reacción humano mínimo posible ante un estímulo auditivo/visual es de **120 ms**. Cualquier registro menor a esto representa un fallo de captura o ruido del sensor y debe eliminarse.

### 3.2. Tratamiento de Outliers con IQR
Para los tiempos de reacción inusualmente lentos, aplicamos la técnica del Rango Intercuartílico (IQR) para filtrar la cola derecha de la distribución sin sesgar los datos principales.

In [ ]:
# 3.1. Limpieza neurofisiológica (< 120ms)
df_clean = df[df['reaction_time_ms'] >= 120]

# 3.2. Outliers lentos mediante IQR en reaction_time_ms
Q1 = df_clean['reaction_time_ms'].quantile(0.25)
Q3 = df_clean['reaction_time_ms'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

df_clean = df_clean[df_clean['reaction_time_ms'] <= upper_limit]

# 3.3. Eliminación de variables de ruido o sesgo
columns_to_drop = ['record_id', 'player_id', 'match_outcome']
df_clean = df_clean.drop(columns=columns_to_drop, errors='ignore')

print(f"Registros post-limpieza: {df_clean.shape[0]} (Se eliminaron {df.shape[0] - df_clean.shape[0]} registros anómalos).")

---
# 4. Fase: Modelamiento (Modeling)

### 4.1. División en Entrenamiento y Prueba
Dividimos el dataset limpio en **80% de entrenamiento** (para ajustar los modelos) y **20% de prueba** (para evaluar su capacidad de generalización ante datos no vistos), asegurando reproducibilidad con una semilla fija (`random_state=42`).

### 4.2. Pipeline y Preprocesamiento Avanzado
Para evitar el *data leakage* (fuga de datos) y lograr un código limpio, implementamos un `ColumnTransformer` que:
- Codifica variables categóricas usando `OneHotEncoder`.
- Estandariza variables numéricas usando `StandardScaler`.

In [ ]:
# 4.1. Separación de Variables (Features y Target)
X = df_clean.drop(columns=['performance_score'])
y = df_clean['performance_score']

# 4.2. División Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Conjunto de Entrenamiento: {X_train.shape[0]} registros.")
print(f"Conjunto de Prueba: {X_test.shape[0]} registros.")

In [ ]:
# 4.3. Definición de Preprocesadores
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

print("Variables numéricas a estandarizar:", num_features)
print("Variables categóricas a codificar:", cat_features)

In [ ]:
# 4.4. Creación y Entrenamiento de los Pipelines
pipelines = {
    'Regresión Lineal': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]),
    'Árbol de Decisión': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeRegressor(random_state=42, max_depth=8))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(random_state=42, n_estimators=100, max_depth=12))
    ])
}

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    print(f"Entrenamiento exitoso para el modelo: {name}")

---
# 5. Fase: Evaluación (Evaluation)

Calculamos las métricas fundamentales para medir la calidad del ajuste y la capacidad de generalización en datos nuevos:
- **Coeficiente de Determinación ($R^2$):** Proporción de la varianza explicada por el modelo.
- **MAE (Error Absoluto Medio):** Promedio de los errores absolutos.
- **MSE (Error Cuadrático Medio):** Promedio de los errores al cuadrado.
- **RMSE (Raíz del Error Cuadrático Medio):** Raíz cuadrada del MSE (en las mismas unidades del target).

In [ ]:
# 5.1. Evaluación e Interpretación de Métricas
results = []

for name, pipeline in pipelines.items():
    # Predicciones
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    # Métricas de entrenamiento
    r2_train = r2_score(y_train, y_train_pred)
    mae_train = mean_absolute_error(y_train, y_train_pred)
    mse_train = mean_squared_error(y_train, y_train_pred)
    rmse_train = np.sqrt(mse_train)
    
    # Métricas de prueba
    r2_test = r2_score(y_test, y_test_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    mse_test = mean_squared_error(y_test, y_test_pred)
    rmse_test = np.sqrt(mse_test)
    
    results.append({
        'Modelo': name,
        'R2 Train': r2_train,
        'R2 Test': r2_test,
        'MAE Train': mae_train,
        'MAE Test': mae_test,
        'MSE Train': mse_train,
        'MSE Test': mse_test,
        'RMSE Train': rmse_train,
        'RMSE Test': rmse_test
    })

df_results = pd.DataFrame(results)
df_results.style.format({
    'R2 Train': '{:.4f}', 'R2 Test': '{:.4f}',
    'MAE Train': '{:.2f}', 'MAE Test': '{:.2f}',
    'MSE Train': '{:.2f}', 'MSE Test': '{:.2f}',
    'RMSE Train': '{:.2f}', 'RMSE Test': '{:.2f}'
})

### 5.2. Comparación y Justificación de Resultados

*(Espacio para documentar el análisis comparativo una vez que se ejecuten los modelos, argumentando el nivel de generalización del mejor algoritmo y la importancia de las métricas en este dominio competitivo).*

---
# 6. Fase: Despliegue (Deployment)

### Formulario Interactivo para Predicciones en Tiempo Real
Utilizando el mejor modelo entrenado, creamos un formulario interactivo con `ipywidgets` para ingresar datos de un nuevo registro y obtener de forma inmediata el score de rendimiento predicho.

In [ ]:
# Seleccionamos el mejor modelo automáticamente (ej. el de mayor R2 en Test)
best_model_name = df_results.loc[df_results['R2 Test'].idxmax(), 'Modelo']
best_pipeline = pipelines[best_model_name]

print(f"Modelo seleccionado para despliegue: {best_model_name}")

# 6.1. Componentes del Formulario
style = {'description_width': 'initial'}

widget_team = widgets.Dropdown(options=df_clean['team_name'].unique().tolist(), description='Equipo:', style=style)
widget_role = widgets.Dropdown(options=df_clean['player_role'].unique().tolist(), description='Rol del Jugador:', style=style)
widget_map = widgets.Dropdown(options=df_clean['map_played'].unique().tolist(), description='Mapa Jugado:', style=style)
widget_match = widgets.Dropdown(options=df_clean['match_type'].unique().tolist(), description='Fase del Torneo:', style=style)

widget_kills = widgets.IntSlider(value=15, min=0, max=50, step=1, description='Kills:', style=style)
widget_assists = widgets.IntSlider(value=8, min=0, max=50, step=1, description='Assists:', style=style)
widget_deaths = widgets.IntSlider(value=10, min=0, max=50, step=1, description='Deaths:', style=style)

widget_accuracy = widgets.FloatSlider(value=45.0, min=0.0, max=100.0, step=0.1, description='Precisión (%):', style=style)
widget_reaction = widgets.FloatSlider(value=200.0, min=120.0, max=500.0, step=1.0, description='Reacción (ms):', style=style)
widget_fatigue = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01, description='Índice Fatiga:', style=style)
widget_win_prob = widgets.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.01, description='Prob. Victoria:', style=style)
widget_mvp = widgets.Dropdown(options=['Yes', 'No'], description='MVP Award:', style=style)

btn_predict = widgets.Button(description='Predecir Desempeño', button_style='primary', icon='calculator')
out_result = widgets.Output()

def handle_prediction(b):
    with out_result:
        clear_output()
        # Creación del dataframe con la entrada
        input_data = pd.DataFrame([{
            'team_name': widget_team.value,
            'player_role': widget_role.value,
            'map_played': widget_map.value,
            'match_type': widget_match.value,
            'kills': widget_kills.value,
            'assists': widget_assists.value,
            'deaths': widget_deaths.value,
            'accuracy_percent': widget_accuracy.value,
            'reaction_time_ms': widget_reaction.value,
            'fatigue_index': widget_fatigue.value,
            'win_probability': widget_win_prob.value,
            'mvp_award': widget_mvp.value
        }])
        
        # Predicción usando el pipeline preentrenado
        prediction = best_pipeline.predict(input_data)[0]
        
        print("\n" + "="*45)
        print(f"🎯 Score de Rendimiento Predicho: {prediction:.2f}")
        print("="*45)

btn_predict.on_click(handle_prediction)

# Organizar la UI
box_categorical = widgets.VBox([widget_team, widget_role, widget_map, widget_match, widget_mvp])
box_numerical = widgets.VBox([widget_kills, widget_assists, widget_deaths, widget_accuracy, widget_reaction, widget_fatigue, widget_win_prob])
ui_form = widgets.HBox([box_categorical, box_numerical])

display(widgets.HTML("<h3>Formulario de Predicción en Tiempo Real</h3>"))
display(ui_form)
display(btn_predict)
display(out_result)